In [0]:
%sql
CREATE OR REPLACE TABLE gold_dev.unified.county_unified AS
SELECT
  counties.county_fips,
  counties.county_name,
  states.state_abbreviation,
  county_population.population_estimate_2025,
  ruca.avg_primary_ruca,
  ruca.land_area,
  ruca.population_density,
  poverty_estimates.perc_of_pop_in_poverty_2023,
  IFNULL(hospitals.ct_hospitals, 0) AS ct_hospitals,
  hospitals.avg_hospital_rating,
  IFNULL(hospitals.ct_hospitals, 0) / county_population.population_estimate_2025 * 100000 
    AS hospitals_per_100_000
FROM
  silver_dev.geo.counties AS counties
  LEFT JOIN silver_dev.demographics.county_population AS county_population
  ON counties.county_fips = county_population.county_fips
  LEFT JOIN silver_dev.geo.states AS states 
  ON counties.state_fips = states.state_fips
  LEFT JOIN gold_dev.demographics.ruca_county AS ruca
  ON counties.county_fips = ruca.county_fips
  LEFT JOIN (
      SELECT county_fips, 
             attribute_value / 100.0 AS perc_of_pop_in_poverty_2023
      FROM silver_dev.demographics.poverty_estimates
      WHERE attribute_name = 'PCTPOVALL_2023'
  ) AS poverty_estimates
  ON counties.county_fips = poverty_estimates.county_fips
  LEFT JOIN gold_dev.healthcare.county_hospital_summary AS hospitals
  ON counties.county_fips = hospitals.county_fips
WHERE states.is_territory = 0
ORDER BY county_fips

In [0]:
%sql
SELECT * FROM gold_dev.unified.county_unified